Cleaning for Npc

In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv(r"C:\Users\abba\Desktop\DS Projects\AMAC-Rental-Intelligence-Dashboard\Data\raw\npc_raw_listings.csv")
df.head()

,Property Type,Price (Per Annum),Location
0,3 bedroom semi-detached duplex for rent,"₦32,000,000","Olu Awotesu Street, Jabi, Abuja"
1,3 bedroom flat / apartment for rent,"₦8,500,000","Games Village, Kaura, Abuja"
2,1 bedroom mini flat (room and parlour) for rent,"₦1,500,000","Along Apo Primary/wumba, Apo, Abuja"
3,3 bedroom flat / apartment for rent,"₦10,000,000","Apo Nepa By Aa Rano Filling Station, Apo, Abuja"
4,Office space for rent,"₦1,300,000","Utako, Abuja"


In [ ]:
#Create bedrooms column by extracting number of bedrooms from property type column using regex
df['Bedrooms'] = df['Property Type'].str.extract(r"(\d+)")
df['Bedrooms'] = df['Bedrooms'].astype('Int64')
df.dropna(subset= "Bedrooms", inplace = True)
df.head()

,Property Type,Price (Per Annum),Location,Bedrooms
0,3 bedroom semi-detached duplex for rent,"₦32,000,000","Olu Awotesu Street, Jabi, Abuja",3
1,3 bedroom flat / apartment for rent,"₦8,500,000","Games Village, Kaura, Abuja",3
2,1 bedroom mini flat (room and parlour) for rent,"₦1,500,000","Along Apo Primary/wumba, Apo, Abuja",1
3,3 bedroom flat / apartment for rent,"₦10,000,000","Apo Nepa By Aa Rano Filling Station, Apo, Abuja",3
5,2 bedroom flat / apartment for rent,"₦7,500,000","Apo Nepa By A.a Rano Filling Station, Apo, Abuja",2


In [4]:
# clean up the property type column by removing numbers and any irrelevant text(for rent, bedroom) from the column
# Define patterns to remove
patterns = [r'\d+\s*bedroom', r'for rent']
# Replace all patterns with an empty string, then clean up whitespace
df["Property Type"] = df["Property Type"].str.replace('|'.join(patterns), '', regex=True, case=False).str.strip()
df.head(12)

,Property Type,Price (Per Annum),Location,Bedrooms
0,semi-detached duplex,"₦32,000,000","Olu Awotesu Street, Jabi, Abuja",3
1,flat / apartment,"₦8,500,000","Games Village, Kaura, Abuja",3
2,mini flat (room and parlour),"₦1,500,000","Along Apo Primary/wumba, Apo, Abuja",1
3,flat / apartment,"₦10,000,000","Apo Nepa By Aa Rano Filling Station, Apo, Abuja",3
5,flat / apartment,"₦7,500,000","Apo Nepa By A.a Rano Filling Station, Apo, Abuja",2
6,detached duplex,"₦9,000,000","Around Sunnyvale Estate, Lokogoma District, Abuja",5
7,flat / apartment,"₦3,500,000","Apo, Abuja",3
8,terraced duplex,"₦12,000,000","Mabushi, Abuja",4
9,house,"₦8,000,000","Jahi, Abuja",3
10,flat / apartment,"₦20,000,000","Life Camp, Abuja",2


In [5]:
df['Property Type'].unique()

array(['semi-detached duplex', 'flat / apartment',
       'mini flat (room and parlour)', 'detached duplex',
       'terraced duplex', 'house', 'detached bungalow',
       'semi-detached bungalow', 'terraced bungalow', 'office space',
       'restaurant / bar', 'commercial property', 'block of flats',
       'hotel / guest house'], dtype=object)

In [8]:
#drop non residential property types from dataset
df = df[~df["Property Type"].str.contains("office space|restaurant / bar|commercial property|hotel / guest house", case=False)]
df['Property Type'].unique()

array(['semi-detached duplex', 'flat / apartment',
       'mini flat (room and parlour)', 'detached duplex',
       'terraced duplex', 'house', 'detached bungalow',
       'semi-detached bungalow', 'terraced bungalow', 'block of flats'],
      dtype=object)

In [14]:
print(df['Property Type'].value_counts())

Property Type
flat / apartment                1633
terraced duplex                 1010
detached duplex                  582
semi-detached duplex             557
house                            195
mini flat (room and parlour)     153
detached bungalow                 87
semi-detached bungalow            25
block of flats                    10
terraced bungalow                  2
Name: count, dtype: int64


In [18]:
# Example logic: If bedrooms > 3, it's likely a duplex. If <= 3, it's a bungalow.
#redefine house rows for a better fit 
def redefine_house(row):
    if row['Property Type'] == 'house':
        if row['Bedrooms'] >= 4:
            return 'detached duplex'
        else:
            return 'detached bungalow'
    return row['Property Type']

df['Property Type'] = df.apply(redefine_house, axis=1)
df['Property Type'].value_counts()

Property Type
flat / apartment                1633
terraced duplex                 1010
detached duplex                  653
semi-detached duplex             557
detached bungalow                211
mini flat (room and parlour)     153
semi-detached bungalow            25
block of flats                    10
terraced bungalow                  2
Name: count, dtype: int64

In [19]:
#normalizing property types
prop_types = {
    'semi-detached duplex': 'Duplex',
    'detached duplex': 'Duplex',
    'terraced duplex': 'Duplex',
    'flat / apartment': 'Apartment',
    'mini flat (room and parlour)': 'Apartment',
    'block of flats': 'Apartment',
    'detached bungalow': 'Bungalow',
    'semi-detached bungalow': 'Bungalow',
    'terraced bungalow': 'Bungalow',
}
# Apply mapping to create a normalized column
df["Property Category"] = df["Property Type"].map(prop_types)

# Check results
df[["Property Type", "Property Category"]].drop_duplicates()
df.head()


,Property Type,Price (Per Annum),Location,Bedrooms,Property Category
0,semi-detached duplex,"₦32,000,000","Olu Awotesu Street, Jabi, Abuja",3,Duplex
1,flat / apartment,"₦8,500,000","Games Village, Kaura, Abuja",3,Apartment
2,mini flat (room and parlour),"₦1,500,000","Along Apo Primary/wumba, Apo, Abuja",1,Apartment
3,flat / apartment,"₦10,000,000","Apo Nepa By Aa Rano Filling Station, Apo, Abuja",3,Apartment
5,flat / apartment,"₦7,500,000","Apo Nepa By A.a Rano Filling Station, Apo, Abuja",2,Apartment


In [21]:
df.shape

(4254, 5)

In [22]:
#Clean Price (Per Annum), by converting dollar listings into naira
# Step 1: Define exchange rate (10th feb, 2026 12:00pm)
usd_to_naira = 1354.97

# Step 2: Function to convert from dollar to naira
def convert_price(price):
    price = str(price).strip()
    
    # Handle USD listings
    if price.startswith("$"):
        amount_usd = float(price.replace("$", "").replace(",", ""))
        amount_naira = amount_usd * usd_to_naira
    
    # Handle existing Naira listings (cleaning them for consistency)
    else:
        # Remove '₦' and commas, then convert to float
        amount_naira = float(price.replace("₦", "").replace(",", ""))
    
    # Return everything in a consistent ₦ format
    return f"₦{amount_naira:,.2f}"

# Step 3: Apply to a NEW column
# This keeps 'Price (Per Annum)' exactly as it was
df["Price (Converted to Naira)"] = df["Price (Per Annum)"].apply(convert_price)

df[['Price (Per Annum)', 'Price (Converted to Naira)']].head()

,Price (Per Annum),Price (Converted to Naira)
0,"₦32,000,000","₦32,000,000.00"
1,"₦8,500,000","₦8,500,000.00"
2,"₦1,500,000","₦1,500,000.00"
3,"₦10,000,000","₦10,000,000.00"
5,"₦7,500,000","₦7,500,000.00"


In [23]:
df["Price(Float)"] = ( df["Price (Converted to Naira)"] .str.replace("₦", "", regex=False) )
df[["Price (Converted to Naira)", "Price(Float)"]].head()

,Price (Converted to Naira),Price(Float)
0,"₦32,000,000.00","32,000,000.00"
1,"₦8,500,000.00","8,500,000.00"
2,"₦1,500,000.00","1,500,000.00"
3,"₦10,000,000.00","10,000,000.00"
5,"₦7,500,000.00","7,500,000.00"


In [24]:
#create district column 
import numpy as np
amac_districts = ['Jabi', 'Kaura', 'Garki', 'Kabusa', 'City Centre', 'Wuse', 'Gwarinpa', 'Gui', 'Karshi', 'Asokoro', 'Jahi', 'Guzape', 'Apo', 'Durumi', 'Lugbe', 'Lokogoma', 'Maitama', 'Wuye', 'Katampe', 'Life Camp', 'Utako', 'Mabushi', 'Idu', 'Kado']

# Step 1: Extract district name from Location
def extract_district(location):
    if pd.isna(location):
        return np.nan
    for district in amac_districts:
        if district.lower() in location.lower():
            return district
    return np.nan

df["District"] = df["Location"].apply(extract_district)

# Step 2: Drop rows where District is NaN (not in AMAC list)
df = df.dropna(subset=["District"])

df[["Location", "District"]].head()

,Location,District
0,"Olu Awotesu Street, Jabi, Abuja",Jabi
1,"Games Village, Kaura, Abuja",Kaura
2,"Along Apo Primary/wumba, Apo, Abuja",Apo
3,"Apo Nepa By Aa Rano Filling Station, Apo, Abuja",Apo
5,"Apo Nepa By A.a Rano Filling Station, Apo, Abuja",Apo


In [25]:
df.head()

,Property Type,Price (Per Annum),Location,Bedrooms,Property Category,Price (Converted to Naira),Price(Float),District
0,semi-detached duplex,"₦32,000,000","Olu Awotesu Street, Jabi, Abuja",3,Duplex,"₦32,000,000.00","32,000,000.00",Jabi
1,flat / apartment,"₦8,500,000","Games Village, Kaura, Abuja",3,Apartment,"₦8,500,000.00","8,500,000.00",Kaura
2,mini flat (room and parlour),"₦1,500,000","Along Apo Primary/wumba, Apo, Abuja",1,Apartment,"₦1,500,000.00","1,500,000.00",Apo
3,flat / apartment,"₦10,000,000","Apo Nepa By Aa Rano Filling Station, Apo, Abuja",3,Apartment,"₦10,000,000.00","10,000,000.00",Apo
5,flat / apartment,"₦7,500,000","Apo Nepa By A.a Rano Filling Station, Apo, Abuja",2,Apartment,"₦7,500,000.00","7,500,000.00",Apo


In [27]:
df.to_csv(r"C:\Users\abba\Desktop\DS Projects\AMAC-Rental-Intelligence-Dashboard\Data\cleaned\amac_rental_cleaned_v1.csv", index =False, )

In [28]:
#drop columns
df= df.drop(['Property Type', 'Price (Per Annum)', 'Location','Price (Converted to Naira)'], axis=1)
df.head()

,Bedrooms,Property Category,Price(Float),District
0,3,Duplex,"32,000,000.00",Jabi
1,3,Apartment,"8,500,000.00",Kaura
2,1,Apartment,"1,500,000.00",Apo
3,3,Apartment,"10,000,000.00",Apo
5,2,Apartment,"7,500,000.00",Apo


In [29]:
#removing commas and converting price(float) column into a float data type
df["Price(Float)"] = ( df["Price(Float)"] .str.replace(",", "", regex=False) ).astype(float)
df.head()

,Bedrooms,Property Category,Price(Float),District
0,3,Duplex,32000000.0,Jabi
1,3,Apartment,8500000.0,Kaura
2,1,Apartment,1500000.0,Apo
3,3,Apartment,10000000.0,Apo
5,2,Apartment,7500000.0,Apo


In [30]:
df.to_csv(r"C:\Users\abba\Desktop\DS Projects\AMAC-Rental-Intelligence-Dashboard\Data\cleaned\amac_rental_cleaned_v2.csv", index =False, )